In [ ]:
D!pip install -q nltk scikit-learn tensorflow transformers datasets
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df = pd.read_csv("https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv")
df['label'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)
def clean_text(text):
 text = text.lower()
 text = re.sub(r'<.*?>', '', text)
 text = re.sub(r'[^a-z\s]', '', text)
 words = text.split()
 words = [w for w in words if w not in stop_words]
 return ' '.join(words)
df['clean_review'] = df['review'].apply(clean_text)
X_train, X_test, y_train, y_test = train_test_split(df['clean_review'], df['label'],
test_size=0.2, random_state=42)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#B. TF-IDF + Logistic Regression
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
lr = LogisticRegression()
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Classification Report:\n", classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.8876
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.87      0.89      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [ ]:
#C. LSTM Deep Learning Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
max_len = 200
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
model = Sequential([
 Embedding(10000, 64, input_length=max_len),
 LSTM(64),
 Dropout(0.3),
 Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_pad, y_train, epochs=3, validation_split=0.2, batch_size=128)
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"LSTM Accuracy: {acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 104s 388ms/step - accuracy: 0.5098 - loss: 0.6927 - val_accuracy: 0.5110 - val_loss: 0.6916
Epoch 2/3
 10/250 ━━━━━━━━━━━━━━━━━━━━ 1:03 266ms/step - accuracy: 0.5385 - loss: 0.6892

In [ ]:
#D. BERT Implementation
!pip install --upgrade transformers
import transformers
import importlib
importlib.reload(transformers)
from transformers import BertTokenizerFast, TFAutoModelForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
train_encodings = bert_tokenizer(list(X_train), truncation=True, padding=True,
max_length=128, return_tensors='tf')
test_encodings = bert_tokenizer(list(X_test), truncation=True, padding=True,
max_length=128, return_tensors='tf')
bert_model = TFAutoModelForSequenceClassification.from_pretrained('bert-base-uncased',
num_labels=2)
bert_model.compile(optimizer=Adam(learning_rate=2e-5),
loss='sparse_categorical_crossentropy', metrics=['accuracy'])
bert_model.fit(
 x={'input_ids': train_encodings['input_ids'], 'attention_mask':
train_encodings['attention_mask']},
 y=np.array(y_train),
 validation_split=0.1,
 epochs=2,
 batch_size=16,
 callbacks=[EarlyStopping(patience=1)]
)
results = bert_model.evaluate(
 x={'input_ids': test_encodings['input_ids'], 'attention_mask':
test_encodings['attention_mask']},
 y=np.array(y_test)
 )
print(f"BERT Accuracy: {results[1]:.4f}")